# 🏆 The ML Gold Standard: Business-First Engineering Manifesto
Version 3.0 — Focus: Continuous Prediction & XGBoost Excellence
Effective Date: March 8, 2026

This notebook serves as the technical implementation of my professional standards for Regression and Forecasting. It is designed to transform complex, multi-dimensional data into precise, actionable continuous values.

Core Philosophy: I solve real-world business problems by applying rigorous engineering standards to messy data. I don't just predict categories; I quantify the future to drive high-stakes decision-making.

The Regression Pillars
Leakage Prevention: Strict separation of training and validation to ensure real-world reliability.

XGBoost Optimization: Utilizing gradient boosting to capture non-linear relationships that linear models miss.

Residual Rigor: Evaluating models not just on accuracy, but on error distribution and variance stability.

Stakeholder Transparency: Translating "Black Box" feature importance into clear business drivers.

In [ ]:
# COMMANDMENT 0: INFRASTRUCTURE SETUP
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
import xgboost as xgb  # Added XGBoost
from sklearn.model_selection import train_test_split, KFold, cross_validate # Changed to KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge  # Glass-box Baseline
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# Create Manifesto Directory Structure
for folder in ['data', 'visuals', 'Presentation'] :
    os.makedirs(folder, exist_ok=True)

print("✅ Regression Infrastructure Ready: data/, visuals/, Presentation/")

In [ ]:
# COMMANDMENT 1: THE WALL OF SILENCE
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']

# Rule: Standard split for continuous targets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Data Split Complete. Train shape: {X_train.shape}, Test shape: {X_test.shape}")

In [ ]:
# COMMANDMENT 2: MISSINGNESS AS A SIGNAL
def add_missing_indicators(X_df):
    X_copy = X_df.copy()
    for col in X_copy.columns:
        if X_copy[col].isnull().sum() > 0:
            X_copy[f'{col}_is_missing'] = X_copy[col].isnull().astype(int)
    return X_copy

X_train = add_missing_indicators(X_train)
X_test = add_missing_indicators(X_test)

# COMMANDMENT 3: THE TRANSFORMATION ENGINE
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

print("✅ Pipeline Preprocessor defined for Regression.")

In [ ]:
# COMMANDMENT 4 & 5: BASELINE VS SOTA & KPI SELECTION
# Using Neg Mean Absolute Error because it's intuitive for business stakeholders
SCORING_METRIC = 'neg_mean_absolute_error'

# 1. Simple Baseline (Ridge Regression - easy to explain coefficients)
baseline_model = Pipeline(steps=[('pre', preprocessor), ('m', Ridge())])

# 2. Complex Model (XGBoost - State of the Art)
complex_model = Pipeline(steps=[
    ('pre', preprocessor),
    ('m', xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42))
])

print(f"✅ Models ready. Optimizing for: {SCORING_METRIC}")

In [ ]:
# COMMANDMENT 6: STABILITY CHECK
def validate_stability(model, name):
    # Standard KFold for regression
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=SCORING_METRIC)

    # We take the absolute because sklearn returns negative values for error metrics
    mean_s = np.abs(scores['test_score'].mean())
    std_s = scores['test_score'].std()

    print(f"🚀 {name} MAE: {mean_s:.4f} (+/- {std_s:.4f})")
    return mean_s

b_score = validate_stability(baseline_model, "Baseline")
c_score = validate_stability(complex_model, "XGBoost")

# Relative Improvement (Lower error is better)
improvement = (b_score - c_score) / b_score
print(f"📈 Relative Error Reduction: {improvement:.2%}")

In [ ]:
# COMMANDMENT 7 & 8: ERROR ANALYSIS
complex_model.fit(X_train, y_train)
y_pred = complex_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Final Test MAE: {mae:.4f}")
print(f"Final Test R² Score: {r2:.4f}")

# Residual Plot
plt.figure(figsize=(10, 6))
residuals = y_test - y_pred
sns.scatterplot(x=y_pred, y=residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.title("Post-Mortem: Residual Analysis (Errors)")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals (Actual - Predicted)")
plt.savefig('visuals/regression_error_analysis.png')
plt.show()

In [ ]:
# COMMANDMENT 9: FEATURE IMPORTANCE (INTERPRETABILITY)
# Extract the model from the pipeline
final_model = complex_model.named_steps['m']
# Get feature names after preprocessing (especially if OneHotEncoding was used)
ohe_features = complex_model.named_steps['pre'].transformers_[1][1].get_feature_names_out(categorical_features)
all_features = np.concatenate([numeric_features, ohe_features])

# Create Importance DataFrame
importances = pd.DataFrame({
    'feature': all_features,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

# Visualization
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importances, palette='magma')
plt.title("Top 10 Business Drivers (XGBoost Feature Importance)")
plt.xlabel("Importance Score")
plt.ylabel("Feature Name")
plt.tight_layout()
plt.savefig('visuals/feature_importance.png')
plt.show()

print("✅ Feature importance mapped. Top driver identified.")

In [ ]:
# COMMANDMENT 10: VALIDATION LAYER (STAKEHOLDER DASHBOARD)
def quick_predict(**kwargs):
    row = pd.DataFrame([kwargs])
    for col in X_train.columns:
        if col not in row.columns: row[col] = 0

    # Ensure column order matches training
    row = row[X_train.columns]
    pred = complex_model.predict(row)[0]
    print(f"--- Predicted Value: {pred:.4f} ---")

# Automatically select the Top 5 most important features for the UI
top_features = importances['feature'].values[:5]

ui_elements = {f: widgets.FloatSlider(
    min=float(X_train[f].min()),
    max=float(X_train[f].max()),
    value=float(X_train[f].mean()),
    description=f[:15]
) for f in top_features}

print("🎮 INTERACTIVE REGRESSION DASHBOARD (TOP DRIVERS)")
widgets.interact(quick_predict, **ui_elements);